# Baselines A and B on ARID (Kaggle)

Seventh Kaggle session, third domain: **ARID v1**, actions filmed in the dark. All runs use the
XOV ARID test list: 823 videos, 11 classes, 6 familiar / 5 unfamiliar. Every setting is the one
already used on UCF101, and nothing is tuned. With only 11 classes, accuracy intervals are wide.
The source groups are assumed from file names (`arid_field2_assumed_session`) and affect only
the intervals.

1. **Gate:** frozen CLIP ViT-B/32 on 2 videos per class (22 videos).
2. **Baseline A, frozen CLIP:** ViT-B/32 and ViT-B/16, 1 and 28 prompt templates, common protocol.
3. **Baseline B, Open-VCLIP adapter:** native configuration (8×3×1) and common protocol. Open-VCLIP
   publishes no ARID test script, so there is no official-script comparison here. The adapter's
   agreement with the official code is measured on UCF101 (and on HMDB51 in the fifth session).

TC-CLIP (Baseline C) on ARID runs in the TC-CLIP notebook.

**Before running**

1. Settings → Accelerator: **GPU T4**. Settings → Internet: **On**.
2. No input needed. Section 3 downloads the authors' ARID v1 release (`ARID_v1_200630.zip`,
   2.45 GB, CC BY 4.0) from the link on xuyu0010.github.io/arid.html and records its SHA-256.
3. Save Version → **Save & Run All (Commit)**. Download **`ovar_outputs_7.zip`** from the Output tab.

Rough time: setup and download 15 min, frozen CLIP about 20 min, Open-VCLIP about 20 min.

In [ ]:
import os, subprocess, sys, json, hashlib, shutil
from pathlib import Path

REPO = "eylulpelinkilic/generalized-category-discovery"
REF = "main"            # replace with a commit SHA to pin the code version
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
PROJECT = CODE / "action_recognition"
RUN_CLIP = True         # 2. frozen CLIP, four configurations
RUN_OVC = True          # 3. Open-VCLIP adapter, native and common
ZIP = "ovar_outputs_7.zip"
STATUS = {}


def run(cmd, cwd=None, allow_fail=False, log=None):
    """Run a shell command, stream (and optionally save) its output, raise on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (PROJECT if PROJECT.exists() else WORK),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    sink = open(log, "a") if log else None
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-15:]
        if sink:
            sink.write(line)
    code = p.wait()
    if sink:
        sink.close()
    if code and not allow_fail:
        last = "\n".join(tail)
        raise RuntimeError(f"command failed with exit code {code}: {cmd}\n--- last output lines ---\n{last}")
    return code


def step(name, fn, needs=(), enabled=True):
    """Run one stage; record a failure and continue, so independent later stages still run."""
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if not enabled:
        STATUS[name] = "not requested"
        return
    if missing:
        STATUS[name] = f"skipped: needs {', '.join(missing)}"
        print(f"-- {name} skipped (needs {missing})")
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

## 1. Code: clone the repository

In [ ]:
if subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}", REF],
                  capture_output=True, text=True).returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Settings -> Internet on (phone-verified account), "
                       "then restart the session.")

if not CODE.exists():
    CODE.mkdir(parents=True)
    run("git init -q && git remote add origin https://github.com/" + REPO + ".git", cwd=CODE)
run(f"git fetch -q --depth 1 origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit:", COMMIT)
assert (PROJECT / "kaggle" / "ovar_arid_baselines.ipynb").exists(), \
    "the cloned code predates this notebook: push the latest commit to GitHub (or set REF)"

## 2. Environment (this repository)

In [ ]:
run("pip install -q open_clip_torch ftfy regex")
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")
run("ffmpeg -version | head -1")

import torch, open_clip
print("python", sys.version.split()[0], "| torch", torch.__version__, "| open_clip", open_clip.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU"

run("python -m ovar fetch-repos")
run("python -m unittest discover -s tests -t .")

## 3. Data: the official ARID v1 release

An attached copy (any input containing `Drink/Drink_10_1.mp4`) is used if present. Otherwise the
authors' compact v1.0 zip is downloaded, its SHA-256 recorded, and `clips_v1/` extracted. All 823
listed test clips were checked to be in this zip, by its file list. Validation then checks every
listed file and decodes 100 of them.

In [ ]:
ARID_URL = ("https://drive.usercontent.google.com/download?id=1s1fenc8fDEXCIHFDv_eI0Fa42xisff3w"
            "&export=download&confirm=t")
ARID_BYTES = 2_450_783_151           # ARID_v1_200630.zip: the authors' compact ARID v1.0 release
ARID_DIR = Path("/tmp/arid")         # ephemeral disk, not the 20 GB /kaggle/working output


def find_arid():
    found = [p.parent.parent for depth in range(5)
             for p in Path("/kaggle/input").glob("*/" * depth + "Drink/Drink_10_1.mp4")]
    return sorted(found, key=lambda p: len(p.parts))


def download_arid():
    root = ARID_DIR / "clips_v1"
    archive_file = ARID_DIR / "ARID_v1_200630.zip"
    if (root / "Drink").is_dir() and not archive_file.exists():
        return root, json.loads((ARID_DIR / "archive.json").read_text())
    ARID_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):         # curl resumes a partial file with -C -
        if archive_file.exists() and archive_file.stat().st_size == ARID_BYTES:
            break
        run(f"curl -sS -L --fail --retry 5 --retry-delay 10 -C - -o {archive_file} '{ARID_URL}'", allow_fail=True)
    size = archive_file.stat().st_size if archive_file.exists() else 0
    if size != ARID_BYTES:
        raise RuntimeError(f"ARID download incomplete: {size} of {ARID_BYTES} bytes. If Google Drive refuses "
                           "(download quota), download ARID_v1_200630.zip in a browser (xuyu0010.github.io/arid.html, "
                           "original version, compact), upload it as a private Kaggle dataset and attach it.")
    archive = {"url": ARID_URL, "bytes": size, "sha256": sha256(archive_file),
               "release": "ARID v1.0, compact zip (ARID_v1_200630.zip) from xuyu0010.github.io/arid.html",
               "licence": "CC BY 4.0"}
    run(f"unzip -q -o {archive_file} 'clips_v1/*' -d {ARID_DIR}")
    (ARID_DIR / "archive.json").write_text(json.dumps(archive) + "\n")
    archive_file.unlink()
    return root, archive


candidates = find_arid()
if candidates:
    ARID_ROOT = candidates[0]
    DATA_SOURCE = {"source": "kaggle input", "path": str(ARID_ROOT),
                   "other_copies": [str(p) for p in candidates[1:]]}
else:
    print("no attached ARID copy under /kaggle/input; downloading the official ARID v1 release")
    ARID_ROOT, archive = download_arid()
    DATA_SOURCE = {"source": "official ARID v1 release", "path": str(ARID_ROOT), **archive}
DATA_SOURCE["n_mp4"] = sum(1 for _ in ARID_ROOT.glob("*/*.mp4"))
print(json.dumps(DATA_SOURCE, indent=2))
(PROJECT / "runs").mkdir(exist_ok=True)
(PROJECT / "runs" / "DATA_SOURCE.json").write_text(json.dumps(DATA_SOURCE, indent=2) + "\n")

CKPT_DIR = Path("/tmp/checkpoints")              # outside /kaggle/working: not part of the outputs
OVC_CKPT = CKPT_DIR / "openvclip-b16" / "swa_2_22.pth"
CLIP_JIT = Path.home() / ".cache" / "clip" / "ViT-B-16.pt"
(PROJECT / "configs" / "paths.yaml").write_text(
    "dataset_roots:\n"
    f"  arid: {ARID_ROOT}\n"
    "checkpoints:\n"
    f"  openvclip_vitb16: {OVC_CKPT}\n"
    f"  clip_vitb16_openai_pt: {CLIP_JIT}\n"
    "cache_dir: cache\n"
    "runs_dir: runs\n")
print((PROJECT / "configs" / "paths.yaml").read_text())
run("python -m ovar validate --datasets arid --check-files --check-decode --decode-limit 100")

## 4. Weights

* open_clip ViT-B/32 and ViT-B/16 for frozen CLIP (same files as session 1).
* Open-VCLIP `swa_2_22.pth`: its SHA-256 must equal sessions 3–4's (docs/backbones.md: `49a5bf45…`).
* Original OpenAI `ViT-B-16.pt`, used to build Open-VCLIP and for weight patching. Its URL contains
  its SHA-256, which is verified.

In [ ]:
OVC_URL = ("https://drive.usercontent.google.com/download?id=1QLdkVYPP_AjFJgB9mGNLhppqvHFWkT4x"
           "&export=download&confirm=t")
OVC_BYTES = 598_587_149
OVC_SHA = "49a5bf452d044f18e8a3c2dff63d46986aeb082be22efe74b70df28cfe08e570"   # sessions 3-4
CLIP_JIT_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_JIT_URL = f"https://openaipublic.azureedge.net/clip/models/{CLIP_JIT_SHA}/ViT-B-16.pt"
CLIP_JIT_BYTES = 350_837_078
DOWNLOADS = {}


def fetch(url, dest, nbytes):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not (dest.exists() and dest.stat().st_size == nbytes):
        run(f"curl -sS -L --fail --retry 5 -C - -o '{dest}' '{url}'")
    size = dest.stat().st_size
    if size != nbytes:
        raise RuntimeError(f"{dest.name}: got {size} bytes, expected {nbytes}. If Google Drive refuses "
                           "(download quota), download the file in a browser, upload it as a private "
                           f"Kaggle dataset, and copy it to {dest}.")
    return {"url": url, "bytes": size, "sha256": sha256(dest)}


def clip_weights():
    run("python -m ovar fetch-clip --arch ViT-B-32 --yes")
    run("python -m ovar fetch-clip --arch ViT-B-16 --yes")


def ovc_weights():
    DOWNLOADS["openvclip_vitb16"] = fetch(OVC_URL, OVC_CKPT, OVC_BYTES)
    DOWNLOADS["clip_vitb16_openai_pt"] = fetch(CLIP_JIT_URL, CLIP_JIT, CLIP_JIT_BYTES)
    print(json.dumps(DOWNLOADS, indent=2))
    (PROJECT / "runs" / "DOWNLOADS.json").write_text(json.dumps(DOWNLOADS, indent=2) + "\n")
    if DOWNLOADS["clip_vitb16_openai_pt"]["sha256"] != CLIP_JIT_SHA:
        raise RuntimeError("ViT-B-16.pt SHA-256 does not match its official URL")
    if DOWNLOADS["openvclip_vitb16"]["sha256"] != OVC_SHA:
        raise RuntimeError("swa_2_22.pth differs from the checkpoint checked in sessions 3-4")


step("clip_weights", clip_weights)
step("ovc_weights", ovc_weights, enabled=RUN_OVC)

## 5. Gate: small run on 2 videos per class

New dataset on real videos: first a seeded 22-video run with frozen CLIP ViT-B/32. The full runs
below need it to succeed.

In [ ]:
def gate():
    run("python -m ovar infer clip_vitb32__common_16x1x1 --per-class 2 --set 'datasets=[arid]' "
        "--variant arid_subset2 --n-boot 200")
    failed = json.loads((PROJECT / "runs" / "clip_vitb32__common_16x1x1__arid_subset2" / "run.json")
                        .read_text())["datasets"]["arid"]["failed"]
    if failed:
        raise RuntimeError(f"{failed} of 22 videos failed to decode: see failures.csv before the full runs")


step("gate", gate, needs=["clip_weights"])

## 6. Baseline A: frozen CLIP on the full ARID test list

The 28-template runs reuse the cached video embeddings of the 1-template runs.

In [ ]:
CLIP_EXPERIMENTS = ["clip_vitb32__common_16x1x1", "clip_vitb32__common_16x1x1_ens28",
                    "clip_vitb16__common_16x1x1", "clip_vitb16__common_16x1x1_ens28"]
for exp in CLIP_EXPERIMENTS:
    step(exp, lambda exp=exp: run(f"python -m ovar infer {exp} --set 'datasets=[arid]' --variant arid"),
         needs=["gate"], enabled=RUN_CLIP)

## 7. Baseline B: Open-VCLIP adapter, native configuration and common protocol

The checkpoint is a pickle, hence `model.trust_pickle=true`. It comes from the official Drive
folder, and its SHA-256 equals sessions 3–4's.

In [ ]:
NATIVE_RUN = PROJECT / "runs" / "openvclip_vitb16__openvclip_native_8x3x1__arid"
for exp in ["openvclip_vitb16__openvclip_native_8x3x1", "openvclip_vitb16__common_16x1x1"]:
    step(exp, lambda exp=exp: run(f"python -m ovar infer {exp} --set 'datasets=[arid]' "
                                  "--set model.trust_pickle=true --variant arid"),
         needs=["gate", "ovc_weights"], enabled=RUN_OVC)

## 8. Reports and outputs

In [ ]:
step("report", lambda: run("python -m ovar report"))
if STATUS.get("report") == "ok":
    from IPython.display import Markdown, display
    display(Markdown((PROJECT / "reports" / "results.md").read_text()))

(PROJECT / "runs" / "CODE_COMMIT.txt").write_text(COMMIT + "\n")
(PROJECT / "runs" / "NOTEBOOK_STATUS.json").write_text(json.dumps(STATUS, indent=2) + "\n")
run(f"rm -f {WORK}/{ZIP} && cd {PROJECT} && zip -qr {WORK}/{ZIP} runs reports configs/paths.yaml")
run(f"ls -lh {WORK}/{ZIP}")

In [ ]:
# Does not raise: an error here could cost the saved outputs of a committed run.
failed = {k: v for k, v in STATUS.items() if v not in ("ok", "not requested")}
print("=" * 70)
print("STAGES NOT COMPLETED - send the zip or this output:" if failed else "all requested stages completed")
for name, state in STATUS.items():
    print(f"{name:42s} {state}")
print("=" * 70)